# 🐍 1 — File Handling + Serialization & Deserialization

**Welcome to Week 4: Advanced Python!** 🎉 Until now, everything our programs made disappeared the moment the program closed. **File handling** lets us **save data permanently** — into files on the disk — and read it back later.

Every topic is in **simple English**, with **real-life examples** and **line-by-line comments**.

---

## 📚 What you will learn here

| # | Topic |
|---|-------|
| 1 | Some theory (text vs binary files) |
| 2 | Writing to a text file (`w`, `a` modes) |
| 3 | How `open()` works + why we close files |
| 4 | Reading from files (`read`, `readline`) |
| 5 | The `with` context manager |
| 6 | Cursor position: `tell()` and `seek()` |
| 7 | Reading big files in chunks |
| 8 | Working with binary files & other data types |
| 9 | **Serialization / Deserialization** (JSON) |
| 10 | **Pickling** custom objects |

---

## 1️⃣ Some Theory

### Two types of data for Input/Output
- **Text** — like `"12345"` stored as a sequence of readable (Unicode) characters.
- **Binary** — like `12345` stored as raw bytes (0s and 1s).

### Two types of files
| Type | Examples |
|------|----------|
| **Text files** | `.txt`, `.py`, `.csv` — all program/source files |
| **Binary files** | images, music, video, `.exe` files |

### How file I/O works (in almost every language)
1. **Open** a file
2. **Read / Write** data
3. **Close** the file

## 2️⃣ Writing to a Text File

To work with a file we use the built-in **`open()`** function. It needs two things:
1. The **file name** (or full path).
2. The **mode** — `"w"` to **write**, `"r"` to **read**, `"a"` to **append**.

After writing, we **close** the file with `.close()`.

In [ ]:
# Case 1 — the file does not exist yet, so Python creates it
f = open("sample.txt", "w")   # open in WRITE mode
f.write("Hello world")        # write some text into it
f.close()                     # always close when done

### ⚠️ You can't use a file after closing it

In [ ]:
f.write("Hello")   # ❌ ValueError: I/O operation on closed file (we already closed it)

### Writing multiple lines
Use `\n` (newline) to move to the next line inside the file.

In [ ]:
f = open("sample1.txt", "w")
f.write("Hello world")
f.write("\nhow are you")
f.close()

### ⚠️ `"w"` mode ERASES the old content

If you open an **existing** file in `"w"` mode and write to it, all the old content is **deleted** and replaced with the new content.

In [ ]:
# The old "Hello world" is wiped and replaced
f = open("sample.txt", "w")
f.write("Salman bhai")
f.close()

## 3️⃣ How Does `open()` Work?

A file normally lives on the **hard drive (ROM)** — permanent storage. When you call `open()`, Python **loads that file into RAM** in a temporary space called a **buffer**.

- While the file is open, all your read/write operations happen in this fast **buffer memory**.
- When you `close()` the file, Python saves everything back to the hard drive and removes it from RAM.

🧠 **Analogy:** the file is a book on a shelf (hard drive). `open()` brings it to your desk (RAM) so you can read/write. `close()` puts it back on the shelf.

### The problem with `"w"` mode → the fix is `"a"` (append)

`"w"` erases everything. If you want to **keep** the old content and **add** new content at the end, use **append mode `"a"`**.

In [ ]:
# "a" = append -> keeps old content, adds new text at the end
f = open("sample1.txt", "a")
f.write("\ni am fine")
f.close()

> 📌 **Remember:**
> - **`"w"` (write)** → erases old content.
> - **`"a"` (append)** → keeps old content, adds to the end.

### Writing many lines at once with `writelines()`
If you have a **list** of strings, `writelines()` writes them all in one go.

In [ ]:
l = ["hello", "\nhi", "\nhow are you", "\ni am fine"]
f = open("sample.txt", "w")
f.writelines(l)   # write the whole list in one line of code
f.close()

### Why do we always close a file? 🔒

Two important reasons:
1. **Memory:** an open file stays in RAM. With big datasets (files in GBs), forgetting to close wastes memory. `close()` frees it.
2. **Safety:** while open, the file's buffer is exposed. Closing it protects the data from unwanted access.

## 4️⃣ Reading From a File

There are two main functions:
- **`read()`** → reads the **whole** file at once (as one string).
- **`readline()`** → reads **one line at a time**.

> 📝 **Fixed from the original:** several cells wrote `f.close` (without `()`), which does **not** actually close the file — I corrected them to `f.close()`.

In [ ]:
f = open("sample.txt", "r")   # "r" = read mode
s = f.read()                  # read everything as one string
print(s)
f.close()

hello
hi
how are you
i am fine


> 💡 **Important:** text files only understand **strings**. Numbers, lists, and dicts are all treated as text — files do not understand other data types directly.

### Read only part of a file with `read(n)`
`read(10)` reads only the **first 10 characters**.

In [ ]:
f = open("sample.txt", "r")
s = f.read(10)   # read just the first 10 characters
print(s)
f.close()

hello
hi
h


### Read line by line with `readline()`

In [ ]:
f = open("sample.txt", "r")
print(f.readline(), end="")   # first line
print(f.readline(), end="")   # second line
f.close()

hello
hi


### Reading ALL lines with a loop

`readline()` is great for **big files** — it reads one line at a time instead of loading the whole file into memory. But how do we know when to stop? When `readline()` returns an **empty string `""`**, we've reached the end.

In [ ]:
f = open("sample.txt", "r")
while True:
    data = f.readline()
    if data == "":     # empty string means end of file
        break
    print(data, end="")
f.close()

hello
hi
how are you
i am fine


> 📌 **Rule of thumb:** big file → use `readline()` (line by line). Small file → `read()` is fine.

## 5️⃣ The `with` Context Manager

Closing a file every time is easy to forget. The **`with`** keyword closes the file **automatically** as soon as its block is done — cleaner and safer.

In [ ]:
# with automatically closes the file — no need for f.close()
with open("sample1.txt", "w") as f:
    f.write("salman bhai")

In [ ]:
# reading with 'with'
with open("sample.txt", "r") as f:
    print(f.read())

hello
hi
how are you
i am fine


## 6️⃣ Cursor Position — the Buffer Remembers

The buffer keeps a **cursor** that remembers how far you've read. So if you call `read(10)` twice, the second call reads the **next** 10 characters, not the same ones.

In [ ]:
with open("sample.txt", "r") as f:
    print(f.read(10))   # first 10 characters
    print(f.read(10))   # the NEXT 10 characters

hello
hi
h
ow are you


### `tell()` — where is the cursor now?
`tell()` reports the current cursor position (how many characters have been read).

In [ ]:
with open("sample.txt", "r") as f:
    print(f.read(10))   # read 10 characters
    print(f.tell())     # cursor is now at position 10

hello
hi
h
10


### `seek()` — move the cursor anywhere
`seek(n)` jumps the cursor to position `n`. `seek(0)` sends it back to the start.

In [ ]:
with open("sample.txt", "r") as f:
    f.seek(15)          # jump to position 15
    print(f.read(10))
    print(f.tell())     # 25
    f.seek(0)           # back to the beginning
    print(f.read(10))   # reads the first 10 again
    print(f.tell())     # 10

e you
i am
25
hello
hi
h
10


### `seek()` during writing
When writing, `seek(0)` moves back and **overwrites** from there (it does not erase the rest). `write()` returns the number of characters written.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("Hello")
    f.seek(0)             # go back to the start
    print(f.write("X"))   # overwrites 'H' with 'X' -> file becomes "Xello"

1


## 7️⃣ Loading a Big File in Chunks

Imagine a **10 GB** file but only **8 GB** of RAM — you can't load it all at once. The solution: read it in small **chunks** using `read(chunk_size)` in a loop. Each chunk is read, used, and discarded, so RAM only ever holds a little at a time.

> 🛠️ **Fixed from the original:** the original chunk loop called `read()` several times per iteration, which **skipped** data. Here is a clean, correct version.

In [ ]:
# make a big file first
big_l = ["hello world " for i in range(1000)]
with open("big.txt", "w") as f:
    f.writelines(big_l)

# read it 100 characters at a time
with open("big.txt", "r") as f:
    chunk_size = 100
    while True:
        data = f.read(chunk_size)   # read the next chunk
        if data == "":              # empty means end of file
            break
        print(data, end="")

hello world hello world hello world hello world hello world ...(the whole file, printed 100 characters at a time)


## 8️⃣ Problems With Text Mode

Text mode has two limits:
1. It **can't handle binary files** (like images).
2. It **can't store other data types** (int, float, dict) directly.

### Binary files need `"rb"` / `"wb"`

Trying to read an image in text mode fails:
```python
with open("photo.jpeg", "r") as f:
    f.read()   # ❌ UnicodeDecodeError — images aren't text
```
Use **`"rb"`** (read binary) and **`"wb"`** (write binary) instead. Here we copy an image byte-for-byte:

In [ ]:
# Copy an image file using binary mode (needs an actual image named photo.jpeg)
with open("photo.jpeg", "rb") as f:          # read binary
    with open("photo_copy.jpeg", "wb") as wf:  # write binary
        wf.write(f.read())                     # read and write the raw bytes

### Other data types must be converted to strings

A text file can only store **strings**. Writing a number directly fails.

In [ ]:
with open("sample.txt", "w") as f:
    f.write(5)   # ❌ TypeError: write() argument must be str, not int

In [ ]:
# convert the number to a string first
with open("sample.txt", "w") as f:
    f.write("5")   # works now

And when you read a number back, it comes back as a **string** — you must convert it before doing maths.

In [ ]:
with open("sample.txt", "r") as f:
    print(int(f.read()) + 5)   # convert the string "5" to int, then add -> 10

10


### The real problem — complex data types (dict)

You can turn a dict into a string with `str()` and save it. But when you read it back, it's just a **string** — you **cannot** easily turn it back into a real dict. The structure is lost.

In [ ]:
d = {"name": "nitish", "age": 20, "gender": "male"}
with open("sample.txt", "w") as f:
    f.write(str(d))   # save it as text

In [ ]:
with open("sample.txt", "r") as f:
    content = f.read()
    print(content)
    print(type(content))   # it's a str, NOT a dict — the structure is gone

{'name': 'nitish', 'age': 20, 'gender': 'male'}
<class 'str'>


So once a dict goes into a text file, you can't get it back as a usable dict. 😩 **This exact problem is solved by Serialization & Deserialization.**

## 9️⃣ Serialization & Deserialization (JSON)

- **Serialization** = converting a Python data type **into** JSON format (to save it).
- **Deserialization** = converting JSON **back into** a Python data type (to use it).

### What is JSON?

**JSON** (JavaScript Object Notation) is a **universal text format** that *every* programming language understands. Almost every API on the internet today uses JSON to send data.

```
   Python object              JSON text file              Python object
   {"name": "nitish"}  ──►  {"name": "nitish"}  ──►   {"name": "nitish"}
        (dict)         dump      (on disk)      load        (dict)
      SERIALIZE                              DESERIALIZE
```

We use Python's built-in **`json`** module: `json.dump()` to save, `json.load()` to read back.

In [ ]:
# SERIALIZATION — save a list to a JSON file
import json
l = [1, 2, 3, 4]
with open("demo.json", "w") as f:
    json.dump(l, f)   # write the list as JSON

In [ ]:
# save a dict — 'indent=4' makes the file nicely formatted and readable
d = {"name": "nitish", "age": 20, "gender": "male"}
with open("demo.json", "w") as f:
    json.dump(d, f, indent=4)

In [ ]:
# DESERIALIZATION — read it back as a real dict (no manual conversion needed!)
import json
with open("demo.json", "r") as f:
    data = json.load(f)   # JSON -> dict
    print(data)
    print(type(data))     # it's a real dict again!

{'name': 'nitish', 'age': 20, 'gender': 'male'}
<class 'dict'>


🎉 That's the magic — the dict comes back as a **real dict**, ready to use. Use JSON whenever you store complex data types (list, dict, etc.).

> 📌 **One quirk:** a **tuple** gets stored as a JSON array, so it comes back as a **list** (not a tuple). That's usually fine.

In [ ]:
# tuples come back as lists after a JSON round-trip
import json
t = (1, 2, 3, 4, 5)
with open("demo.json", "w") as f:
    json.dump(t, f)
with open("demo.json", "r") as f:
    data = json.load(f)
print(data)
print(type(data))   # a list, not a tuple

[1, 2, 3, 4, 5]
<class 'list'>


### Serializing a custom object

JSON knows Python's built-in types, but **not your own classes**. If you try to `json.dump()` a custom object directly, you get an error:
```
TypeError: Object of type Person is not JSON serializable
```

The fix: give `json.dump()` a **`default=`** function that tells it how to turn your object into a dict.

> 🛠️ **Fixed from the original:** one cell had `def show_object(person)` **without a colon** (a syntax error) — corrected below.

In [ ]:
class Person:
    def __init__(self, fname, lname, age, gender):
        self.fname = fname
        self.lname = lname
        self.age = age
        self.gender = gender

person = Person("Nitish", "singh", 33, "male")

In [ ]:
import json

# this function tells json HOW to convert a Person object into a dict
def show_object(person):
    if isinstance(person, Person):
        return {"name": person.fname + " " + person.lname,
                "age": person.age,
                "gender": person.gender}

with open("demo.json", "w") as f:
    json.dump(person, f, default=show_object, indent=4)   # default= does the trick

In [ ]:
# read the custom object back (as a dict)
import json
with open("demo.json", "r") as f:
    data = json.load(f)
    print(data)
    print(type(data))

{'name': 'Nitish singh', 'age': 33, 'gender': 'male'}
<class 'dict'>


## 🔟 Pickling — Saving an Object *As It Is*

With JSON, a custom object comes back as a **dict** — it **loses its powers** (you can't call its methods anymore).

**Pickling** solves this. It converts a whole Python object into a **byte stream** (binary) and saves it. When you load it back (**unpickling**), you get the **exact same object** — with all its attributes **and methods** working.

| Term | Meaning |
|------|---------|
| **Pickling** | object → byte stream (binary file) |
| **Unpickling** | byte stream → the exact same object |

🧠 So you can save an object, send the file to another computer, load it there, and it behaves exactly like the original. We use the built-in **`pickle`** module.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age
    def display_info(self):
        print("Hi my name is", self.name, "and i am", self.age, "years old")

p = Person("nitish", 33)

In [ ]:
# PICKLE DUMP — save the whole object to a binary file
import pickle
with open("person.pkl", "wb") as f:   # note: wb (write binary)
    pickle.dump(p, f)

In [ ]:
# UNPICKLE (load) — get the exact same object back, methods and all
import pickle
with open("person.pkl", "rb") as f:   # rb (read binary)
    p = pickle.load(f)
    p.display_info()   # the method still works! -> the object kept its powers

Hi my name is nitish and i am 33 years old


### Pickle vs JSON — which to use?

| | **JSON** | **Pickle** |
|--|----------|------------|
| Format | human-readable **text** | **binary** (not readable) |
| Works with | built-in types (list, dict...) | any Python object |
| Custom objects | come back as a **dict** (lose methods) | come back as the **full object** (keep methods) |
| Sharing across languages | ✅ universal | ❌ Python only |

> 💡 **Use JSON** when you need readable data or must share it with other systems (APIs). **Use Pickle** when you need to save/restore a complete Python object with its behaviour intact.

---

## 🎯 Summary — Quick Revision

You can now save and load data permanently! 🎉

| Concept | Key idea |
|---------|----------|
| **`open(name, mode)`** | Modes: `w` (write/erase), `a` (append), `r` (read), `rb`/`wb` (binary) |
| **Always close** | Use `.close()`, or better, the `with` block (auto-closes) |
| **`read` / `readline`** | Whole file vs line-by-line (line-by-line is better for big files) |
| **`tell` / `seek`** | Find / move the cursor position |
| **Chunks** | `read(chunk_size)` in a loop to handle huge files |
| **Text limits** | Only stores strings; can't hold binary or complex types directly |
| **JSON** | `dump`/`load` complex data as universal, readable text |
| **Pickle** | `dump`/`load` a whole object (binary) — keeps its methods |

---

### 📝 Try It Yourself (Practice Questions)

1. Write your name and age to a file, then read them back and print a greeting.
2. Append 3 new lines to an existing file without erasing the old ones.
3. Count the number of lines in a text file using a loop.
4. Save a dictionary of 3 students and their marks to a JSON file, then read it back.
5. Create a `Book` object, pickle it to a file, then unpickle it and call one of its methods.

> 💪 **Remember:** file handling is how real programs remember things. Every app that saves your data (games, notes, settings) uses these exact ideas!

---

**➡️ Next up: 2 — Exception Handling, Modules & Packages.**